In [16]:
!pip install pillow
!pip install chardet
!pip install pytesseract
!apt install tesseract-ocr -y
!pip install pytesseract
!apt-get install tesseract-ocr
!pip install pytesseract

In [7]:
#Read image (png) file
from PIL import Image
def load_png_file(image_path):
    try:
        image = Image.open(image_path)
        print("Image file loaded successfully.")
        return image
    except Exception as e:
        print("Error loading image file:", e)
        return None

# import from our file
image_object = load_png_file("/content/sample_1_image.png")

print("\n--- Image Info ---")
print("Format:", image_object.format)
print("Size:", image_object.size)
print("Mode:", image_object.mode)


Image file loaded successfully.

--- Image Info ---
Format: PNG
Size: (1023, 625)
Mode: RGBA


In [9]:
#read text file (function)
import chardet

def load_text_file(file_path):
    with open(file_path, "rb") as file:
        raw_data = file.read()
        result = chardet.detect(raw_data)
        encoding = result['encoding']

    with open(file_path, "r", encoding=encoding) as file:
        text = file.read()

    print(f"Text file loaded using detected encoding: {encoding}")
    return text


In [10]:
#Text file output text
text_df = load_text_file("/content/sample_1_text.txt")
text_df

Text file loaded using detected encoding: Windows-1252


'INTRODUCTION\nTHIS document is a template for Microsoft Word versions 6.0 or later. If you are reading a paper version of this document, please download the electronic file, ieeeconf_letter.dot (for letter sized paper: 8.5” x 11”) or ieeeconf_A4.dot (for A4 sized paper: 210mm x 297mm) and save to MS Word templates directory. The template to produce your conference paper is available at www.paperplaza.net/support/support.html. To create your own document, from within MS Word, open a new document using File | New then select ieeeconf_letter.dot (for letter sized paper) or ieeeconf_A4.dot (for A4 sized paper). All instructions beyond this point are from IEEE. Instructions about final paper and figure submissions in this document are for IEEE journals; please use this document as a “template” to prepare your manuscript. For submission guidelines, follow instructions on paper submission system as well as the Conference website.\nIf your paper is intended for a conference, please contact yo

In [11]:
# Used pytesseract to extract the text from image
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def extract_text_from_image(image_object):
    try:
        text = pytesseract.image_to_string(image_object)
        print("Text extracted from image successfully.")
        return text
    except Exception as e:
        print("Error during OCR:", e)
        return None


# extracted image text from out input file
image_text = extract_text_from_image(image_object)

if image_text:
    print("\n--- Image Text Preview ---")
    print(image_text[:200])


Text extracted from image successfully.

--- Image Text Preview ---
INTRODUCTION

THIS document is a template for Microsoft Word versions 6.0 or later. If you are reading a paper
version of this document, please download the electronic file, ieeeconf_letter.dot (for l


In [17]:
#Word simmilarity using cosin similarity - simple model just looks at words in the text and assign score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def cosine_sim(text1, text2):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([text1, text2])
    return cosine_similarity(vectors[0:1], vectors[1:2])[0][0]

score = cosine_sim(image_text, text_df)

print("Cosine Similarity:", score)
print("Similarity %:", score * 100)


Cosine Similarity: 0.9941357793590809
Similarity %: 99.4135779359081


In [19]:
#Used sentence transformer to understand the sementic relationship wityh the two text
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

emb1 = model.encode(image_text, convert_to_tensor=True)
emb2 = model.encode(text_df, convert_to_tensor=True)

score = util.pytorch_cos_sim(emb1, emb2).item()

print("Semantic Similarity:", score)
print("Similarity %:", score * 100)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Similarity: 0.9996557235717773
Similarity %: 99.96557235717773


In [20]:
#Text file output text
text_df_synomyne = load_text_file("/content/sample_1_text_synomyne.txt")
text_df_synomyne

Text file loaded using detected encoding: Windows-1252


'INTRODUCTION\nThis document serves as a template for Microsoft Word version 6.0 or newer. If you are viewing a printed copy of this file, please obtain the electronic version—either ieeeconf_letter.dot (for letter-size paper: 8.5” × 11”) or ieeeconf_A4.dot (for A4 paper: 210 mm × 297 mm)—and store it in the MS Word templates folder.\nThe template required to prepare your conference manuscript is available at: www.paperplaza.net/support/support.html. To create your document in MS Word, go to File | New, then choose ieeeconf_letter.dot (for letter-size format) or ieeeconf_A4.dot (for A4 format).\nAll guidelines provided after this section originate from IEEE. The directions regarding final manuscript and figure submission included here apply to IEEE journals; use this file strictly as a template when preparing your paper. For submission procedures, follow the instructions provided by the paper submission system as well as the conference website.\nIf your manuscript is intended for a con

In [25]:
#cosin similareity for synomyne text
cosine_score = cosine_sim(image_text, text_df_synomyne)

print("Cosine Similarity:", score)
print("Similarity %:", score * 100)

Cosine Similarity: 0.9648948907852173
Similarity %: 96.48948907852173


In [27]:
#Sementic similarity between synomyne text -- we higher score here as our model is comparing score based on meaning of text and not just words
emb1 = model.encode(image_text, convert_to_tensor=True)
emb2 = model.encode(text_df_synomyne, convert_to_tensor=True)

sementic_score = util.pytorch_cos_sim(emb1, emb2).item()

print("Semantic Similarity:", score)
print("Similarity %:", score * 100)


Semantic Similarity: 0.9648948907852173
Similarity %: 96.48948907852173
